# Feature Engineering — Tricura ML Case

This notebook constructs the **temporal feature matrix** for the Tier 1 ML models:
- **H1:** Falls (7-day horizon)
- **H2:** Return-to-Hospital (7-day horizon)
- **H3:** Wounds / Pressure Injuries (14-day horizon)

### Temporal Design
- **Observation unit:** resident × non-overlapping window (7d for H1/H2, 14d for H3)
- **Prediction gap (embargo):** 1 day — features use data with timestamps ≤ t−1d; labels span [t, t+horizon]
- **Full-signal window:** Jul 2023 – Jan 2025 (vitals coverage)
- **Temporal CV:** 3-fold expanding window + Jan 2025 hold-out

Reference: `eda-findings.md` and `modeling-plan.md`

In [1]:
from pathlib import Path
from datetime import datetime, timedelta

import polars as pl
import numpy as np

DATA_DIR = Path("../data")
RAW = DATA_DIR / "raw"
PROCESSED = DATA_DIR / "processed"
FEATURE_STORE = DATA_DIR / "feature_store"

# ── Temporal Configuration ───────────────────────────────────────
SIGNAL_START = datetime(2023, 7, 1)
SIGNAL_END   = datetime(2025, 2, 1)   # exclusive (data through Jan 2025)

HORIZON_7D  = timedelta(days=7)
EMBARGO_DAYS = 1                       # features use data ≤ t - 1d

# Vitals clipping ranges (from EDA)
VITALS_CLIP = {
    "BP - Systolic": (60, 250),
    "Pulse":         (20, 200),
    "O2 sats":       (50, 100),
    "Blood Sugar":   (20, 600),
    "Temperature":   (90, 108),
    "Respiration":   (5, 60),
    "Pain Level":    (0, 10),
    "Weight":        (50, 500),
}

# Tables to load (PoC: high-coverage sources only)
# Skipping: medications (16%), adl_responses (3.2%), gg_responses (4.5%),
#           therapy_tracks (7.3%) — too sparse for population-level features
TABLES = [
    "residents", "vitals", "incidents", "diagnoses",
    "hospital_transfers", "needs", "care_plans",
    "lab_reports", "physician_orders", "document_tags",
]

print("Configuration ready.")

Configuration ready.


## 1. Load & Clean Raw Data

In [2]:
# Load all tables
raw = {name: pl.read_parquet(RAW / f"{name}.parquet") for name in TABLES}

for name, df in raw.items():
    print(f"{name:25s}  {df.shape[0]:>10,} rows × {df.shape[1]:>2} cols")

residents                       3,000 rows ×  9 cols
vitals                      2,517,056 rows ×  9 cols
incidents                       3,578 rows ×  8 cols
diagnoses                      60,620 rows ×  8 cols
hospital_transfers              1,816 rows × 12 cols
needs                         162,762 rows × 11 cols
care_plans                      3,034 rows ×  9 cols
lab_reports                    13,334 rows ×  9 cols
physician_orders               94,051 rows × 10 cols
document_tags                 562,905 rows ×  9 cols


In [21]:
# ── Residents: active during signal window ────────────────────────
residents = (
    raw["residents"]
    .filter(
        # Admitted before signal window ends
        pl.col("admission_date") < SIGNAL_END,
        # Not discharged before signal window starts (or never discharged)
        (pl.col("discharge_date") >= SIGNAL_START) | pl.col("discharge_date").is_null(),
        # Not deceased before signal window starts (or alive)
        (pl.col("deceased_date") >= SIGNAL_START) | pl.col("deceased_date").is_null(),
    )
    .with_columns(
        # Effective observation start/end per resident
        pl.max_horizontal(pl.col("admission_date"), pl.lit(SIGNAL_START)).alias("obs_start"),
        pl.min_horizontal(
            pl.col("discharge_date").fill_null(pl.lit(SIGNAL_END)),
            pl.col("deceased_date").fill_null(pl.lit(SIGNAL_END)),
            pl.lit(SIGNAL_END),
        ).alias("obs_end"),
    )
)

print(f"Active residents in signal window: {residents.shape[0]:,}")
print(f"Date range: {SIGNAL_START.date()} → {SIGNAL_END.date()}")
display(residents.head(5))

Active residents in signal window: 3,000
Date range: 2023-07-01 → 2025-02-01


resident_id,facility_id,date_of_birth,admission_date,discharge_date,deceased_date,outpatient,created_at,updated_at,obs_start,obs_end
str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],bool,datetime[μs],datetime[μs],datetime[μs],datetime[μs]
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",1936-04-15 00:00:00,2017-10-25 23:00:00,null,null,false,2024-12-17 09:44:08.250,2026-02-13 04:51:18.257,2023-07-01 00:00:00,2025-02-01 00:00:00
"""a13b0271-693e-589a-aac3-8cdbc2…","""018d5b79-b86a-5dc8-8453-417d99…",1953-08-08 00:00:00,2021-09-15 23:00:00,null,null,false,2024-12-17 09:44:28.970,2026-02-10 04:49:02.887,2023-07-01 00:00:00,2025-02-01 00:00:00
"""9a2f39bf-23c2-580b-b17c-8575e8…","""018d5b79-b86a-5dc8-8453-417d99…",1963-07-11 00:00:00,2024-11-28 00:00:00,null,null,false,2024-12-17 09:44:07.793,2026-02-15 08:54:49.493,2024-11-28 00:00:00,2025-02-01 00:00:00
"""8683d1dd-7f25-5d2a-9888-50459d…","""018d5b79-b86a-5dc8-8453-417d99…",1964-07-25 00:00:00,2024-09-17 23:00:00,2024-12-20 20:46:00,null,false,2024-12-17 09:44:43.297,2025-11-20 06:29:39.887,2024-09-17 23:00:00,2024-12-20 20:46:00
"""2b208db4-43fa-5683-9bca-78dcce…","""018d5b79-b86a-5dc8-8453-417d99…",1959-12-27 00:00:00,2024-11-07 23:00:00,2025-03-01 09:58:00,null,false,2024-12-17 09:44:17.947,2025-11-20 06:29:39.887,2024-11-07 23:00:00,2025-02-01 00:00:00


In [22]:
# ── Vitals: filter strikeout, clip outliers, keep signal window ───
vitals = (
    raw["vitals"]
    .filter(
        pl.col("strikeout") == False,
        pl.col("measured_at") >= SIGNAL_START,
        pl.col("measured_at") < SIGNAL_END,
    )
    .select("vital_id", "resident_id", "facility_id", "vital_type", "value", "dystolic_value", "measured_at")
)

# Apply per-type clipping
clip_exprs = []
for vtype, (lo, hi) in VITALS_CLIP.items():
    clip_exprs.append(
        pl.when(pl.col("vital_type") == vtype)
        .then(pl.col("value").clip(lo, hi))
    )

vitals = vitals.with_columns(
    pl.coalesce(clip_exprs + [pl.col("value")]).alias("value")
)

print(f"Vitals after cleaning: {vitals.shape[0]:,} rows")
print(vitals.group_by("vital_type").len().sort("vital_type"))
display(vitals.head(5))

Vitals after cleaning: 2,512,667 rows
shape: (8, 2)
┌───────────────┬────────┐
│ vital_type    ┆ len    │
│ ---           ┆ ---    │
│ str           ┆ u32    │
╞═══════════════╪════════╡
│ BP - Systolic ┆ 370019 │
│ Blood Sugar   ┆ 250258 │
│ O2 sats       ┆ 266527 │
│ Pain Level    ┆ 830599 │
│ Pulse         ┆ 335722 │
│ Respiration   ┆ 189043 │
│ Temperature   ┆ 237982 │
│ Weight        ┆ 32517  │
└───────────────┴────────┘


vital_id,resident_id,facility_id,vital_type,value,dystolic_value,measured_at
str,str,str,str,f64,f64,datetime[μs]
"""4bd6b2ec-b186-550f-a1a8-42016b…","""207a6546-03d7-5343-a214-299291…","""0240d706-3348-5117-8d03-b06c51…","""Pain Level""",0.0,null,2024-12-12 04:31:08
"""168e6d04-8914-54df-8338-f57d70…","""207a6546-03d7-5343-a214-299291…","""0240d706-3348-5117-8d03-b06c51…","""Pain Level""",0.0,null,2024-12-12 04:31:27
"""9d3b01a5-86c6-5e90-bebd-437d6b…","""0c0eada7-fb72-50e3-b33f-3cabeb…","""0240d706-3348-5117-8d03-b06c51…","""Pain Level""",0.0,null,2024-12-12 05:27:24
"""9dec4276-4457-588f-9d6f-31c18a…","""0c0eada7-fb72-50e3-b33f-3cabeb…","""0240d706-3348-5117-8d03-b06c51…","""Pain Level""",8.0,null,2024-12-12 02:37:17
"""3d6d9496-45d1-549e-805a-69cc27…","""725efbae-bafc-53a6-bfa5-ff3ad7…","""0240d706-3348-5117-8d03-b06c51…","""Weight""",179.8,null,2024-08-16 13:17:00


In [26]:
# ── Incidents: active only (strikeout=False) ──────────────────────
incidents = (
    raw["incidents"]
    .filter(pl.col("strikeout") == False)
    .select("incident_id", "resident_id", "facility_id", "incident_type", "occurred_at")
)

# ── Hospital transfers: unplanned only (RTH definition) ──────────
hospital_transfers = (
    raw["hospital_transfers"]
    .filter(
        (pl.col("planned_flag") == False) | pl.col("planned_flag").is_null()
    )
    .select("transfer_id", "resident_id", "facility_id", "effective_date",
            "to_from_type", "transfer_outcome", "transfer_reason", "emergency_flag")
)

# ── Diagnoses: active only ────────────────────────────────────────
diagnoses = (
    raw["diagnoses"]
    .filter(pl.col("strikeout") == False)
    .select("diagnosis_id", "resident_id", "icd_10_code", "onset_at", "resolved_at")
)

# ── Needs: active only ────────────────────────────────────────────
needs = (
    raw["needs"]
    .filter(pl.col("strikeout") == False)
    .select("need_id", "resident_id", "need_category", "initiated_at", "resolved_at")
)

# ── Lab reports ────────────────────────────────────────────────────
lab_reports = (
    raw["lab_reports"]
    .select("lab_report_id", "resident_id", "severity_status", "reported_at")
)

# ── Document tags ──────────────────────────────────────────────────
document_tags = (
    raw["document_tags"]
    .filter(pl.col("deleted_at").is_null())
    .select("document_tag_id", "resident_id", "tag_id", "created_at")
)

print("Incidents:          ", incidents.shape)
print("Hospital transfers: ", hospital_transfers.shape)
print("Diagnoses:          ", diagnoses.shape)
print("Needs:              ", needs.shape)
print("Lab reports:        ", lab_reports.shape)
print("Document tags:      ", document_tags.shape)

Incidents:           (3382, 5)
Hospital transfers:  (1770, 8)
Diagnoses:           (58067, 5)
Needs:               (160466, 5)
Lab reports:         (13334, 4)
Document tags:       (562840, 4)


## 2. Observation Windows & Target Labels

Generate non-overlapping 7-day observation windows per resident.

**Temporal layout** for each window starting at time `t`:
```
◄── Feature lookback ──►│ 1d gap │◄── Label window ──►
                        t-1d      t                   t+7d
```

- **Features** use data with timestamps ≤ `t − 1 day`
- **Labels** check for events in `[t, t + 7d)`

In [30]:
def generate_observation_windows(
    residents_df: pl.DataFrame,
    horizon: timedelta,
) -> pl.DataFrame:
    """
    Generate non-overlapping observation windows for each resident.
    Each window has:
      - window_start (t): beginning of the label window
      - window_end (t + horizon): end of the label window
      - feature_cutoff (t - EMBARGO_DAYS): last date for feature data
    """
    stride_days = horizon.days
    rows = []

    for row in residents_df.iter_rows(named=True):
        rid = row["resident_id"]
        fid = row["facility_id"]
        start = row["obs_start"]
        end = row["obs_end"]

        t = start
        while t + horizon <= end:
            rows.append({
                "resident_id": rid,
                "facility_id": fid,
                "window_start": t,
                "window_end": t + horizon,
                "feature_cutoff": t - timedelta(days=EMBARGO_DAYS),
            })
            t += horizon  # non-overlapping stride

    return pl.DataFrame(rows)


windows_7d = generate_observation_windows(residents, HORIZON_7D)
print(f"7-day observation windows: {windows_7d.shape[0]:,}")
print(f"Unique residents:          {windows_7d['resident_id'].n_unique():,}")
print(f"Window range:              {windows_7d['window_start'].min().date()} → {windows_7d['window_end'].max().date()}")
with pl.Config(set_fmt_str_lengths=1000):
    display(windows_7d.head(5))

7-day observation windows: 66,312
Unique residents:          2,647
Window range:              2023-07-01 → 2025-02-01


resident_id,facility_id,window_start,window_end,feature_cutoff
str,str,datetime[μs],datetime[μs],datetime[μs]
"""578b8ad7-37c2-52b2-8b9e-9d82891ddf41""","""018d5b79-b86a-5dc8-8453-417d99a8f1f9""",2023-07-01 00:00:00,2023-07-08 00:00:00,2023-06-30 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d82891ddf41""","""018d5b79-b86a-5dc8-8453-417d99a8f1f9""",2023-07-08 00:00:00,2023-07-15 00:00:00,2023-07-07 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d82891ddf41""","""018d5b79-b86a-5dc8-8453-417d99a8f1f9""",2023-07-15 00:00:00,2023-07-22 00:00:00,2023-07-14 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d82891ddf41""","""018d5b79-b86a-5dc8-8453-417d99a8f1f9""",2023-07-22 00:00:00,2023-07-29 00:00:00,2023-07-21 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d82891ddf41""","""018d5b79-b86a-5dc8-8453-417d99a8f1f9""",2023-07-29 00:00:00,2023-08-05 00:00:00,2023-07-28 00:00:00


In [31]:
with pl.Config(set_tbl_rows=100):
    display(windows_7d.filter(pl.col("resident_id") == "578b8ad7-37c2-52b2-8b9e-9d82891ddf41").sort("window_start"))

resident_id,facility_id,window_start,window_end,feature_cutoff
str,str,datetime[μs],datetime[μs],datetime[μs]
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-07-01 00:00:00,2023-07-08 00:00:00,2023-06-30 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-07-08 00:00:00,2023-07-15 00:00:00,2023-07-07 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-07-15 00:00:00,2023-07-22 00:00:00,2023-07-14 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-07-22 00:00:00,2023-07-29 00:00:00,2023-07-21 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-07-29 00:00:00,2023-08-05 00:00:00,2023-07-28 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-08-05 00:00:00,2023-08-12 00:00:00,2023-08-04 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-08-12 00:00:00,2023-08-19 00:00:00,2023-08-11 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-08-19 00:00:00,2023-08-26 00:00:00,2023-08-18 00:00:00
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-08-26 00:00:00,2023-09-02 00:00:00,2023-08-25 00:00:00


In [39]:
# ── Target: fall_7d ───────────────────────────────────────────────
falls = incidents.filter(pl.col("incident_type") == "Fall")

fall_labels = (
    windows_7d
    .join(
        falls.select("resident_id", "occurred_at"),
        on="resident_id",
        how="left",
    )
    .filter(
        pl.col("occurred_at").is_null()  # no fall so we keep as negative
        | (
            (pl.col("occurred_at") >= pl.col("window_start"))
            & (pl.col("occurred_at") < pl.col("window_end"))
        )
    )
    .group_by("resident_id", "window_start")
    .agg(
        pl.col("occurred_at").is_not_null().any().cast(pl.Int8).alias("fall_7d"),
    )
)

# ── Target: rth_7d ───────────────────────────────────────────────
rth_labels = (
    windows_7d
    .join(
        hospital_transfers.select("resident_id", "effective_date"),
        on="resident_id",
        how="left",
    )
    .filter(
        pl.col("effective_date").is_null()
        | (
            (pl.col("effective_date") >= pl.col("window_start"))
            & (pl.col("effective_date") < pl.col("window_end"))
        )
    )
    .group_by("resident_id", "window_start")
    .agg(
        pl.col("effective_date").is_not_null().any().cast(pl.Int8).alias("rth_7d"),
    )
)

# ── Merge labels onto windows ────────────────────────────────────
obs = (
    windows_7d
    .join(fall_labels, on=["resident_id", "window_start"], how="left")
    .join(rth_labels, on=["resident_id", "window_start"], how="left")
    .with_columns(
        pl.col("fall_7d").fill_null(0),
        pl.col("rth_7d").fill_null(0),
    )
)

print(f"Observation matrix: {obs.shape[0]:,} rows")
print(f"\nfall_7d distribution:\n{obs['fall_7d'].value_counts().sort('fall_7d')}")
print(f"\nrth_7d distribution:\n{obs['rth_7d'].value_counts().sort('rth_7d')}")
display(obs.head())

Observation matrix: 66,312 rows

fall_7d distribution:
shape: (2, 2)
┌─────────┬───────┐
│ fall_7d ┆ count │
│ ---     ┆ ---   │
│ i8      ┆ u32   │
╞═════════╪═══════╡
│ 0       ┆ 64707 │
│ 1       ┆ 1605  │
└─────────┴───────┘

rth_7d distribution:
shape: (2, 2)
┌────────┬───────┐
│ rth_7d ┆ count │
│ ---    ┆ ---   │
│ i8     ┆ u32   │
╞════════╪═══════╡
│ 0      ┆ 65773 │
│ 1      ┆ 539   │
└────────┴───────┘


resident_id,facility_id,window_start,window_end,feature_cutoff,fall_7d,rth_7d
str,str,datetime[μs],datetime[μs],datetime[μs],i8,i8
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-07-01 00:00:00,2023-07-08 00:00:00,2023-06-30 00:00:00,0,0
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-07-08 00:00:00,2023-07-15 00:00:00,2023-07-07 00:00:00,0,0
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-07-15 00:00:00,2023-07-22 00:00:00,2023-07-14 00:00:00,0,0
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-07-22 00:00:00,2023-07-29 00:00:00,2023-07-21 00:00:00,0,0
"""578b8ad7-37c2-52b2-8b9e-9d8289…","""018d5b79-b86a-5dc8-8453-417d99…",2023-07-29 00:00:00,2023-08-05 00:00:00,2023-07-28 00:00:00,0,0


## 3. Feature Engineering

Feature groups (all point-in-time, using data ≤ `feature_cutoff`):

| Group | Source | Coverage | Features |
|---|---|---|---|
| Demographics | `residents` | 100% | age, LOS, facility |
| Vitals | `vitals` | 82% | rolling mean/std/min/max/slope per type (3d, 7d, 14d) |
| Diagnoses | `diagnoses` | 85% | ICD-10 chapter flags, fall/wound risk codes, comorbidity count |
| Incident history | `incidents` | 33% | prior counts by type (7d, 30d, 90d, all-time) |
| RTH history | `hospital_transfers` | 30% | prior transfer count (30d, 90d, all-time) |
| Care needs | `needs` | 83% | open needs count by category |
| Lab reports | `lab_reports` | 39% | abnormal/critical counts (14d, 30d) |
| Document tags | `document_tags` | 34% | risk-relevant tag flags |

In [35]:
# ── 3a. Demographics ──────────────────────────────────────────────
# Age at window start, length of stay, facility encoding

demo_feats = (
    obs
    .join(
        residents.select("resident_id", "date_of_birth", "admission_date"),
        on="resident_id",
        how="left",
    )
    .with_columns(
        # Age in years at window start
        ((pl.col("window_start") - pl.col("date_of_birth")).dt.total_days() / 365.25)
        .round(1)
        .alias("age_at_window"),

        # Length of stay in days at window start
        ((pl.col("window_start") - pl.col("admission_date")).dt.total_days())
        .alias("los_days"),
    )
    .select("resident_id", "window_start", "age_at_window", "los_days")
)

print("Demographics features:")
display(demo_feats.describe())

Demographics features:


statistic,resident_id,window_start,age_at_window,los_days
str,str,str,f64,f64
"""count""","""66312""","""66312""",66247.0,66312.0
"""null_count""","""0""","""0""",65.0,0.0
"""mean""",null,"""2024-05-05 08:45:16.027868""",75.761603,651.146565
"""std""",null,null,12.973896,820.446723
"""min""","""000d4288-4983-5512-b4e8-bf5efa…","""2023-07-01 00:00:00""",23.6,0.0
"""25%""",null,"""2023-12-16 00:00:00""",67.3,119.0
"""50%""",null,"""2024-05-18 00:00:00""",76.5,388.0
"""75%""",null,"""2024-09-29 08:00:00""",85.9,852.0
"""max""","""fff99568-3ec9-5b97-b909-c0522d…","""2025-01-25 00:00:00""",105.2,7275.0


In [44]:
# ── 3b. Vitals — rolling statistics per vital type ────────────────
# For each observation window, compute stats over 3d/7d/14d lookback
# windows ending at feature_cutoff.
#
# Strategy: join vitals to obs windows, filter by lookback, aggregate.
# This is the most compute-intensive step.

VITAL_LOOKBACKS = [3, 7, 14]  # days
VITAL_AGGS = ["mean", "std", "min", "max"]

def compute_vitals_features(
    obs_df: pl.DataFrame,
    vitals_df: pl.DataFrame,
    lookbacks: list[int] = VITAL_LOOKBACKS,
) -> pl.DataFrame:
    """
    Compute rolling vital statistics for each observation window.
    Uses an asof-style approach: for each (resident, window), pull vitals
    in [feature_cutoff - lookback_days, feature_cutoff].

    Also produces binary `{type}_measured_{lb}d` flags — explicitly encodes
    absence of monitoring as a signal (LightGBM treats null and 0 differently).
    """
    result = obs_df.select("resident_id", "window_start", "feature_cutoff")

    for lb_days in lookbacks:
        lb = timedelta(days=lb_days)
        suffix = f"_{lb_days}d"

        # Join vitals to windows, filter to lookback range
        joined = (
            obs_df.select("resident_id", "window_start", "feature_cutoff")
            .join(
                vitals_df.select("resident_id", "vital_type", "value", "measured_at"),
                on="resident_id",
                how="left",
            )
            .filter(
                pl.col("measured_at").is_not_null()
                & (pl.col("measured_at") <= pl.col("feature_cutoff"))
                & (pl.col("measured_at") > (pl.col("feature_cutoff") - lb))
            )
        )

        # Aggregate per (resident, window, vital_type)
        agg = (
            joined
            .group_by("resident_id", "window_start", "vital_type")
            .agg(
                pl.col("value").mean().alias("mean"),
                pl.col("value").std().alias("std"),
                pl.col("value").min().alias("min"),
                pl.col("value").max().alias("max"),
                pl.col("value").count().alias("count"),
            )
        )

        # Pivot aggregated stats into columns
        pivoted = agg.unpivot(
            index=["resident_id", "window_start", "vital_type"],
            on=["mean", "std", "min", "max", "count"],
            variable_name="stat",
            value_name="val",
        ).with_columns(
            (pl.col("vital_type").str.to_lowercase().str.replace_all(r"[\s\-]", "_")
             + "_" + pl.col("stat") + pl.lit(suffix)).alias("feature_name")
        ).pivot(
            on="feature_name",
            index=["resident_id", "window_start"],
            values="val",
        )

        result = result.join(pivoted, on=["resident_id", "window_start"], how="left")

        # ── Binary "was measured" flags per vital type ────────────
        # Null in the stats columns means zero measurements in the window.
        # An explicit 0/1 flag makes this signal unambiguous for the model.
        measured_flags = (
            agg.select("resident_id", "window_start", "vital_type")
            .with_columns(
                (pl.col("vital_type").str.to_lowercase().str.replace_all(r"[\s\-]", "_")
                 + pl.lit(f"_measured{suffix}")).alias("flag_name"),
                pl.lit(1).cast(pl.Int8).alias("flag_val"),
            )
            .pivot(
                on="flag_name",
                index=["resident_id", "window_start"],
                values="flag_val",
            )
        )

        flag_cols = [c for c in measured_flags.columns if c not in ("resident_id", "window_start")]
        result = (
            result
            .join(measured_flags, on=["resident_id", "window_start"], how="left")
            .with_columns([pl.col(c).fill_null(0) for c in flag_cols])
        )

    return result.drop("feature_cutoff")


vitals_feats = compute_vitals_features(obs, vitals)

vital_cols = [c for c in vitals_feats.columns if c not in ("resident_id", "window_start")]
measured_cols = [c for c in vital_cols if "_measured_" in c]
stat_cols = [c for c in vital_cols if "_measured_" not in c]

print(f"Vitals features: {len(vital_cols)} columns total")
print(f"  Stat features:    {len(stat_cols)}")
print(f"  Measured flags:   {len(measured_cols)}")
print(f"\nMeasured flags: {measured_cols}")
print(f"\nNon-null rates (stat sample):")
for c in stat_cols[:5]:
    rate = 1 - vitals_feats[c].null_count() / vitals_feats.shape[0]
    print(f"  {c}: {rate:.1%}")
print(f"\nMeasured flag rates (% of windows with ≥1 reading):")
for c in measured_cols[:8]:
    rate = vitals_feats[c].mean()
    print(f"  {c}: {rate:.1%}")

Vitals features: 144 columns total
  Stat features:    120
  Measured flags:   24

Measured flags: ['blood_sugar_measured_3d', 'pain_level_measured_3d', 'bp___systolic_measured_3d', 'respiration_measured_3d', 'pulse_measured_3d', 'weight_measured_3d', 'o2_sats_measured_3d', 'temperature_measured_3d', 'pain_level_measured_7d', 'bp___systolic_measured_7d', 'blood_sugar_measured_7d', 'pulse_measured_7d', 'respiration_measured_7d', 'temperature_measured_7d', 'o2_sats_measured_7d', 'weight_measured_7d', 'temperature_measured_14d', 'pain_level_measured_14d', 'pulse_measured_14d', 'bp___systolic_measured_14d', 'weight_measured_14d', 'o2_sats_measured_14d', 'blood_sugar_measured_14d', 'respiration_measured_14d']

Non-null rates (stat sample):
  blood_sugar_mean_3d: 14.9%
  pain_level_mean_3d: 52.0%
  bp___systolic_mean_3d: 37.4%
  respiration_mean_3d: 23.8%
  pulse_mean_3d: 36.3%

Measured flag rates (% of windows with ≥1 reading):
  blood_sugar_measured_3d: 14.9%
  pain_level_measured_3d: 52.

In [48]:
display(vitals_feats.describe())

statistic,resident_id,window_start,blood_sugar_mean_3d,pain_level_mean_3d,bp___systolic_mean_3d,respiration_mean_3d,pulse_mean_3d,weight_mean_3d,o2_sats_mean_3d,temperature_mean_3d,blood_sugar_std_3d,pain_level_std_3d,bp___systolic_std_3d,respiration_std_3d,pulse_std_3d,weight_std_3d,o2_sats_std_3d,temperature_std_3d,blood_sugar_min_3d,pain_level_min_3d,bp___systolic_min_3d,respiration_min_3d,pulse_min_3d,weight_min_3d,o2_sats_min_3d,temperature_min_3d,blood_sugar_max_3d,pain_level_max_3d,bp___systolic_max_3d,respiration_max_3d,pulse_max_3d,weight_max_3d,o2_sats_max_3d,temperature_max_3d,blood_sugar_count_3d,pain_level_count_3d,…,bp___systolic_std_14d,weight_std_14d,o2_sats_std_14d,blood_sugar_std_14d,respiration_std_14d,temperature_min_14d,pain_level_min_14d,pulse_min_14d,bp___systolic_min_14d,weight_min_14d,o2_sats_min_14d,blood_sugar_min_14d,respiration_min_14d,temperature_max_14d,pain_level_max_14d,pulse_max_14d,bp___systolic_max_14d,weight_max_14d,o2_sats_max_14d,blood_sugar_max_14d,respiration_max_14d,temperature_count_14d,pain_level_count_14d,pulse_count_14d,bp___systolic_count_14d,weight_count_14d,o2_sats_count_14d,blood_sugar_count_14d,respiration_count_14d,temperature_measured_14d,pain_level_measured_14d,pulse_measured_14d,bp___systolic_measured_14d,weight_measured_14d,o2_sats_measured_14d,blood_sugar_measured_14d,respiration_measured_14d
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""66312""","""66312""",9911.0,34454.0,24809.0,15809.0,24064.0,8626.0,18029.0,18686.0,8757.0,33359.0,20097.0,9916.0,18734.0,1336.0,13170.0,13313.0,9911.0,34454.0,24809.0,15809.0,24064.0,8626.0,18029.0,18686.0,9911.0,34454.0,24809.0,15809.0,24064.0,8626.0,18029.0,18686.0,9911.0,34454.0,…,28067.0,8419.0,21337.0,10681.0,19639.0,27896.0,36534.0,31855.0,32032.0,25199.0,26041.0,11344.0,25600.0,27896.0,36534.0,31855.0,32032.0,25199.0,26041.0,11344.0,25600.0,27896.0,36534.0,31855.0,32032.0,25199.0,26041.0,11344.0,25600.0,66312.0,66312.0,66312.0,66312.0,66312.0,66312.0,66312.0,66312.0
"""null_count""","""0""","""0""",56401.0,31858.0,41503.0,50503.0,42248.0,57686.0,48283.0,47626.0,57555.0,32953.0,46215.0,56396.0,47578.0,64976.0,53142.0,52999.0,56401.0,31858.0,41503.0,50503.0,42248.0,57686.0,48283.0,47626.0,56401.0,31858.0,41503.0,50503.0,42248.0,57686.0,48283.0,47626.0,56401.0,31858.0,…,38245.0,57893.0,44975.0,55631.0,46673.0,38416.0,29778.0,34457.0,34280.0,41113.0,40271.0,54968.0,40712.0,38416.0,29778.0,34457.0,34280.0,41113.0,40271.0,54968.0,40712.0,38416.0,29778.0,34457.0,34280.0,41113.0,40271.0,54968.0,40712.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,"""2024-05-05 08:45:16.027868""",172.889495,0.565622,127.781932,17.791351,74.70464,174.897013,96.156366,97.695138,43.968117,0.666014,9.53205,0.831335,6.06276,1.102472,1.196008,0.345946,123.746151,0.060574,118.823814,17.259852,69.397985,174.752343,94.998469,97.397887,234.409242,1.688831,137.079447,18.404516,80.468771,175.044089,97.106334,97.978924,7.950661,7.987665,…,10.447833,1.810483,1.328614,43.91787,0.941189,97.157144,0.040264,66.059488,112.660995,171.964194,94.025279,105.401543,16.900703,98.183855,2.762878,85.282963,143.883367,173.078091,97.537529,266.200635,18.981016,11.69458,33.853616,14.814723,16.101867,1.894321,13.793441,31.390779,9.683477,0.420678,0.550941,0.480381,0.48305,0.380007,0.392704,0.17107,0.386054
"""std""",null,null,55.584025,1.147995,12.588867,1.342278,7.855669,59.975008,1.787691,0.467987,28.590888,1.122038,6.508157,1.47524,4.406111,2.190992,1.35961,0.392631,42.597319,0.441202,14.303584,1.44852,8.85663,59.893795,3.323994,0.751355,95.859052,2.817482,16.513156,2.89798,10.838856,60.065724,1.709348,0.664165,5.514177,5.0036,…,5.69624,4.044945,1.497059,26.30508,1.337161,1.06001,0.374281,9.632251,15.023303,57.017415,4

In [49]:
# ── 3c. Diagnosis features (point-in-time snapshot) ──────────────
# Active diagnoses at feature_cutoff: onset_at <= cutoff AND
# (resolved_at > cutoff OR resolved_at IS NULL)
#
# Features:
#   - Total active diagnosis count
#   - ICD-10 chapter flags (first letter/digit)
#   - Specific high-risk code flags relevant to falls & RTH

# High-risk ICD-10 prefixes for falls & RTH
FALL_RISK_CODES = [
    "R26",   # Abnormalities of gait and mobility
    "R27",   # Other lack of coordination
    "M62.81",# Sarcopenia
    "R29.6", # Tendency to fall
    "H81",   # Vestibular disorders
    "G40",   # Epilepsy
    "F01", "F02", "F03",  # Dementia
    "G30",   # Alzheimer's
]

RTH_RISK_CODES = [
    "I50",   # Heart failure
    "J44",   # COPD
    "I10",   # Hypertension
    "E11",   # Type 2 diabetes
    "N18",   # Chronic kidney disease
    "J18",   # Pneumonia
    "I63",   # Cerebral infarction
]

def compute_diagnosis_features(
    obs_df: pl.DataFrame,
    dx_df: pl.DataFrame,
) -> pl.DataFrame:
    """Point-in-time active diagnosis features per observation window."""

    # Join and filter to active diagnoses at feature_cutoff
    active_dx = (
        obs_df.select("resident_id", "window_start", "feature_cutoff")
        .join(dx_df, on="resident_id", how="left")
        .filter(
            pl.col("icd_10_code").is_not_null()
            & (pl.col("onset_at") <= pl.col("feature_cutoff"))
            & (
                pl.col("resolved_at").is_null()
                | (pl.col("resolved_at") > pl.col("feature_cutoff"))
            )
        )
    )

    # Total active diagnosis count
    dx_count = (
        active_dx
        .group_by("resident_id", "window_start")
        .agg(pl.col("icd_10_code").n_unique().alias("dx_active_count"))
    )

    # ICD-10 chapter flags (first character of code)
    # E.g. "I" = Circulatory, "E" = Endocrine, "F" = Mental, "G" = Nervous, etc.
    icd_chapters = (
        active_dx
        .with_columns(pl.col("icd_10_code").str.slice(0, 1).alias("icd_chapter"))
        .group_by("resident_id", "window_start")
        .agg(pl.col("icd_chapter").n_unique().alias("dx_chapter_count"))
    )

    # Fall-risk diagnosis flags
    fall_risk_exprs = []
    for code in FALL_RISK_CODES:
        safe_name = code.replace(".", "_")
        fall_risk_exprs.append(
            pl.col("icd_10_code").str.starts_with(code).any().cast(pl.Int8).alias(f"dx_fall_{safe_name}")
        )

    fall_flags = (
        active_dx
        .group_by("resident_id", "window_start")
        .agg(fall_risk_exprs)
    )
    # Sum into a single fall-risk score
    fall_flag_cols = [c for c in fall_flags.columns if c.startswith("dx_fall_")]
    fall_flags = fall_flags.with_columns(
        pl.sum_horizontal(fall_flag_cols).alias("dx_fall_risk_score")
    )

    # RTH-risk diagnosis flags
    rth_risk_exprs = []
    for code in RTH_RISK_CODES:
        safe_name = code.replace(".", "_")
        rth_risk_exprs.append(
            pl.col("icd_10_code").str.starts_with(code).any().cast(pl.Int8).alias(f"dx_rth_{safe_name}")
        )

    rth_flags = (
        active_dx
        .group_by("resident_id", "window_start")
        .agg(rth_risk_exprs)
    )
    rth_flag_cols = [c for c in rth_flags.columns if c.startswith("dx_rth_")]
    rth_flags = rth_flags.with_columns(
        pl.sum_horizontal(rth_flag_cols).alias("dx_rth_risk_score")
    )

    # Merge all diagnosis features
    result = (
        obs_df.select("resident_id", "window_start")
        .join(dx_count, on=["resident_id", "window_start"], how="left")
        .join(icd_chapters, on=["resident_id", "window_start"], how="left")
        .join(fall_flags, on=["resident_id", "window_start"], how="left")
        .join(rth_flags, on=["resident_id", "window_start"], how="left")
        .with_columns(pl.col("dx_active_count").fill_null(0))
    )

    return result


dx_feats = compute_diagnosis_features(obs, diagnoses)

dx_cols = [c for c in dx_feats.columns if c.startswith("dx_")]
print(f"Diagnosis features: {len(dx_cols)} columns")
print(f"Columns: {dx_cols}")

Diagnosis features: 21 columns
Columns: ['dx_active_count', 'dx_chapter_count', 'dx_fall_R26', 'dx_fall_R27', 'dx_fall_M62_81', 'dx_fall_R29_6', 'dx_fall_H81', 'dx_fall_G40', 'dx_fall_F01', 'dx_fall_F02', 'dx_fall_F03', 'dx_fall_G30', 'dx_fall_risk_score', 'dx_rth_I50', 'dx_rth_J44', 'dx_rth_I10', 'dx_rth_E11', 'dx_rth_N18', 'dx_rth_J18', 'dx_rth_I63', 'dx_rth_risk_score']


In [50]:
display(dx_feats.describe())

statistic,resident_id,window_start,dx_active_count,dx_chapter_count,dx_fall_R26,dx_fall_R27,dx_fall_M62_81,dx_fall_R29_6,dx_fall_H81,dx_fall_G40,dx_fall_F01,dx_fall_F02,dx_fall_F03,dx_fall_G30,dx_fall_risk_score,dx_rth_I50,dx_rth_J44,dx_rth_I10,dx_rth_E11,dx_rth_N18,dx_rth_J18,dx_rth_I63,dx_rth_risk_score
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""66312""","""66312""",66312.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0,50277.0
"""null_count""","""0""","""0""",0.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0,16035.0
"""mean""",null,"""2024-05-05 08:45:16.027868""",13.39373,8.02166,0.372596,0.214651,0.387076,0.09752,0.000895,0.134694,0.067983,0.10651,0.271377,0.08658,1.739881,0.190226,0.195318,0.616624,0.351811,0.166856,0.032182,0.1227,1.675717
"""std""",null,null,11.389111,2.904421,0.483501,0.410584,0.487086,0.296667,0.029904,0.3414,0.25172,0.308492,0.444674,0.281222,1.391558,0.392483,0.396449,0.486213,0.47754,0.372851,0.176484,0.328096,1.212593
"""min""","""000d4288-4983-5512-b4e8-bf5efa…","""2023-07-01 00:00:00""",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",null,"""2023-12-16 00:00:00""",1.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
"""50%""",null,"""2024-05-18 00:00:00""",13.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0
"""75%""",null,"""2024-09-29 08:00:00""",21.0,10.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,2.0
"""max""","""fff99568-3ec9-5b97-b909-c0522d…","""2025-01-25 00:00:00""",82.0,16.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0


In [51]:
# ── 3d. Incident history features ─────────────────────────────────
# Prior incident counts by type and recency window (7d, 30d, 90d, all-time)
# Only uses incidents with occurred_at < feature_cutoff (strict temporal)

HISTORY_LOOKBACKS = {
    "7d":  timedelta(days=7),
    "30d": timedelta(days=30),
    "90d": timedelta(days=90),
}
INCIDENT_TYPES = ["Fall", "Wound", "Altercation"]

def compute_incident_history(
    obs_df: pl.DataFrame,
    incidents_df: pl.DataFrame,
) -> pl.DataFrame:
    """Count prior incidents by type and recency."""

    base = obs_df.select("resident_id", "window_start", "feature_cutoff")

    joined = base.join(
        incidents_df.select("resident_id", "incident_type", "occurred_at"),
        on="resident_id",
        how="left",
    ).filter(
        pl.col("occurred_at").is_not_null()
        & (pl.col("occurred_at") <= pl.col("feature_cutoff"))
    )

    agg_exprs = []

    # All-time counts per type
    for itype in INCIDENT_TYPES:
        safe = itype.lower()
        agg_exprs.append(
            (pl.col("incident_type") == itype).sum().cast(pl.Int32).alias(f"hist_{safe}_all")
        )

    # Lookback-windowed counts per type
    for lb_name, lb_delta in HISTORY_LOOKBACKS.items():
        for itype in INCIDENT_TYPES:
            safe = itype.lower()
            agg_exprs.append(
                (
                    (pl.col("incident_type") == itype)
                    & (pl.col("occurred_at") > (pl.col("feature_cutoff") - lb_delta))
                ).sum().cast(pl.Int32).alias(f"hist_{safe}_{lb_name}")
            )

    # Total incident count (all types)
    agg_exprs.append(pl.len().cast(pl.Int32).alias("hist_incident_total"))

    history = joined.group_by("resident_id", "window_start").agg(agg_exprs)

    result = (
        obs_df.select("resident_id", "window_start")
        .join(history, on=["resident_id", "window_start"], how="left")
    )
    # Fill nulls with 0 for all history columns
    hist_cols = [c for c in result.columns if c.startswith("hist_")]
    result = result.with_columns([pl.col(c).fill_null(0) for c in hist_cols])

    return result


hist_feats = compute_incident_history(obs, incidents)

hist_cols = [c for c in hist_feats.columns if c.startswith("hist_")]
print(f"Incident history features: {len(hist_cols)} columns")
print(f"Columns: {hist_cols}")
print(f"\nResidents with any prior incident: {(hist_feats['hist_incident_total'] > 0).sum():,}")

Incident history features: 13 columns
Columns: ['hist_fall_all', 'hist_wound_all', 'hist_altercation_all', 'hist_fall_7d', 'hist_wound_7d', 'hist_altercation_7d', 'hist_fall_30d', 'hist_wound_30d', 'hist_altercation_30d', 'hist_fall_90d', 'hist_wound_90d', 'hist_altercation_90d', 'hist_incident_total']

Residents with any prior incident: 19,209


In [52]:
# ── 3e. RTH (hospital transfer) history features ─────────────────
# Prior unplanned transfer counts (30d, 90d, all-time)

def compute_rth_history(
    obs_df: pl.DataFrame,
    transfers_df: pl.DataFrame,
) -> pl.DataFrame:
    """Count prior unplanned hospital transfers by recency."""

    base = obs_df.select("resident_id", "window_start", "feature_cutoff")

    joined = base.join(
        transfers_df.select("resident_id", "effective_date", "emergency_flag"),
        on="resident_id",
        how="left",
    ).filter(
        pl.col("effective_date").is_not_null()
        & (pl.col("effective_date") <= pl.col("feature_cutoff"))
    )

    history = (
        joined
        .group_by("resident_id", "window_start")
        .agg(
            pl.len().cast(pl.Int32).alias("hist_rth_all"),
            (pl.col("effective_date") > (pl.col("feature_cutoff") - timedelta(days=30)))
            .sum().cast(pl.Int32).alias("hist_rth_30d"),
            (pl.col("effective_date") > (pl.col("feature_cutoff") - timedelta(days=90)))
            .sum().cast(pl.Int32).alias("hist_rth_90d"),
            # Emergency transfers
            (pl.col("emergency_flag") == True).sum().cast(pl.Int32).alias("hist_rth_emergency_all"),
        )
    )

    result = (
        obs_df.select("resident_id", "window_start")
        .join(history, on=["resident_id", "window_start"], how="left")
    )
    rth_cols = [c for c in result.columns if c.startswith("hist_rth")]
    result = result.with_columns([pl.col(c).fill_null(0) for c in rth_cols])

    return result


rth_hist_feats = compute_rth_history(obs, hospital_transfers)

rth_cols = [c for c in rth_hist_feats.columns if c.startswith("hist_rth")]
print(f"RTH history features: {len(rth_cols)} columns")
print(f"Columns: {rth_cols}")
print(f"Residents with any prior RTH: {(rth_hist_feats['hist_rth_all'] > 0).sum():,}")

RTH history features: 4 columns
Columns: ['hist_rth_all', 'hist_rth_30d', 'hist_rth_90d', 'hist_rth_emergency_all']
Residents with any prior RTH: 12,767


In [53]:
# ── 3f. Care needs features (point-in-time snapshot) ─────────────
# Open needs by category at feature_cutoff

NEED_CATEGORIES = ["Fall", "Wound", "Nutrition", "Other"]

def compute_needs_features(
    obs_df: pl.DataFrame,
    needs_df: pl.DataFrame,
) -> pl.DataFrame:
    """Count open care plan needs by category at feature_cutoff."""

    base = obs_df.select("resident_id", "window_start", "feature_cutoff")

    active_needs = (
        base.join(needs_df, on="resident_id", how="left")
        .filter(
            pl.col("need_category").is_not_null()
            & (pl.col("initiated_at") <= pl.col("feature_cutoff"))
            & (
                pl.col("resolved_at").is_null()
                | (pl.col("resolved_at") > pl.col("feature_cutoff"))
            )
        )
    )

    agg_exprs = [
        pl.len().cast(pl.Int32).alias("needs_open_total"),
    ]
    for cat in NEED_CATEGORIES:
        safe = cat.lower()
        agg_exprs.append(
            (pl.col("need_category") == cat).sum().cast(pl.Int32).alias(f"needs_open_{safe}")
        )

    needs_agg = active_needs.group_by("resident_id", "window_start").agg(agg_exprs)

    result = (
        obs_df.select("resident_id", "window_start")
        .join(needs_agg, on=["resident_id", "window_start"], how="left")
    )
    needs_cols = [c for c in result.columns if c.startswith("needs_")]
    result = result.with_columns([pl.col(c).fill_null(0) for c in needs_cols])

    return result


needs_feats = compute_needs_features(obs, needs)

needs_cols = [c for c in needs_feats.columns if c.startswith("needs_")]
print(f"Needs features: {len(needs_cols)} columns")
print(f"Columns: {needs_cols}")

Needs features: 5 columns
Columns: ['needs_open_total', 'needs_open_fall', 'needs_open_wound', 'needs_open_nutrition', 'needs_open_other']


In [54]:
# ── 3g. Lab report features ───────────────────────────────────────
# Abnormal/critical lab counts in 14d and 30d lookback

def compute_lab_features(
    obs_df: pl.DataFrame,
    labs_df: pl.DataFrame,
) -> pl.DataFrame:
    """Count abnormal/critical labs in recent lookback windows."""

    base = obs_df.select("resident_id", "window_start", "feature_cutoff")

    joined = (
        base.join(
            labs_df.select("resident_id", "severity_status", "reported_at"),
            on="resident_id",
            how="left",
        )
        .filter(
            pl.col("reported_at").is_not_null()
            & (pl.col("reported_at") <= pl.col("feature_cutoff"))
        )
    )

    agg_exprs = []
    for lb_days in [14, 30]:
        lb = timedelta(days=lb_days)
        recent = pl.col("reported_at") > (pl.col("feature_cutoff") - lb)
        agg_exprs.extend([
            (recent & (pl.col("severity_status") == "Abnormal"))
            .sum().cast(pl.Int32).alias(f"labs_abnormal_{lb_days}d"),
            (recent & (pl.col("severity_status") == "Critical"))
            .sum().cast(pl.Int32).alias(f"labs_critical_{lb_days}d"),
            recent.sum().cast(pl.Int32).alias(f"labs_total_{lb_days}d"),
        ])

    labs_agg = joined.group_by("resident_id", "window_start").agg(agg_exprs)

    result = (
        obs_df.select("resident_id", "window_start")
        .join(labs_agg, on=["resident_id", "window_start"], how="left")
    )
    lab_cols = [c for c in result.columns if c.startswith("labs_")]
    result = result.with_columns([pl.col(c).fill_null(0) for c in lab_cols])

    return result


lab_feats = compute_lab_features(obs, lab_reports)

lab_cols = [c for c in lab_feats.columns if c.startswith("labs_")]
print(f"Lab features: {len(lab_cols)} columns")
print(f"Columns: {lab_cols}")

Lab features: 6 columns
Columns: ['labs_abnormal_14d', 'labs_critical_14d', 'labs_total_14d', 'labs_abnormal_30d', 'labs_critical_30d', 'labs_total_30d']


In [55]:
# ── 3h. Document tag features ─────────────────────────────────────
# Risk-relevant clinical tags as binary flags (all-time up to cutoff)

RISK_TAGS = [
    "actual_fall", "fall_risk", "actual_wound", "wound_risk",
    "aggressive_behavior", "alzheimers_disease", "dementia",
    "depression", "anxiety", "pain", "infection",
    "pressure_injury", "skin_tear", "bruise",
    "wandering", "elopement",
]

def compute_tag_features(
    obs_df: pl.DataFrame,
    tags_df: pl.DataFrame,
) -> pl.DataFrame:
    """Binary flags for risk-relevant document tags up to feature_cutoff."""

    base = obs_df.select("resident_id", "window_start", "feature_cutoff")

    relevant_tags = tags_df.filter(pl.col("tag_id").is_in(RISK_TAGS))

    joined = (
        base.join(
            relevant_tags.select("resident_id", "tag_id", "created_at"),
            on="resident_id",
            how="left",
        )
        .filter(
            pl.col("tag_id").is_not_null()
            & (pl.col("created_at") <= pl.col("feature_cutoff"))
        )
    )

    agg_exprs = []
    for tag in RISK_TAGS:
        agg_exprs.append(
            (pl.col("tag_id") == tag).any().cast(pl.Int8).alias(f"tag_{tag}")
        )

    tags_agg = joined.group_by("resident_id", "window_start").agg(agg_exprs)

    result = (
        obs_df.select("resident_id", "window_start")
        .join(tags_agg, on=["resident_id", "window_start"], how="left")
    )
    tag_cols = [c for c in result.columns if c.startswith("tag_")]
    result = result.with_columns([pl.col(c).fill_null(0) for c in tag_cols])

    return result


tag_feats = compute_tag_features(obs, document_tags)

tag_cols = [c for c in tag_feats.columns if c.startswith("tag_")]
print(f"Document tag features: {len(tag_cols)} columns")
print(f"Columns: {tag_cols}")

Document tag features: 16 columns
Columns: ['tag_actual_fall', 'tag_fall_risk', 'tag_actual_wound', 'tag_wound_risk', 'tag_aggressive_behavior', 'tag_alzheimers_disease', 'tag_dementia', 'tag_depression', 'tag_anxiety', 'tag_pain', 'tag_infection', 'tag_pressure_injury', 'tag_skin_tear', 'tag_bruise', 'tag_wandering', 'tag_elopement']


## 4. Assemble Feature Matrix

Join all feature groups onto the observation spine (`obs`). The result is one row per `(resident_id, window_start)` with all features and both target labels.

In [56]:
# ── Merge all feature groups ──────────────────────────────────────
join_keys = ["resident_id", "window_start"]

feature_matrix = (
    obs  # spine: resident_id, window_start, window_end, feature_cutoff, facility_id, fall_7d, rth_7d
    .join(demo_feats, on=join_keys, how="left")
    .join(vitals_feats, on=join_keys, how="left")
    .join(dx_feats, on=join_keys, how="left")
    .join(hist_feats, on=join_keys, how="left")
    .join(rth_hist_feats, on=join_keys, how="left")
    .join(needs_feats, on=join_keys, how="left")
    .join(lab_feats, on=join_keys, how="left")
    .join(tag_feats, on=join_keys, how="left")
)

# Identify column groups
meta_cols = ["resident_id", "facility_id", "window_start", "window_end", "feature_cutoff"]
target_cols = ["fall_7d", "rth_7d"]
feature_cols = [c for c in feature_matrix.columns if c not in meta_cols + target_cols]

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"  Meta columns:    {len(meta_cols)}")
print(f"  Target columns:  {len(target_cols)}")
print(f"  Feature columns: {len(feature_cols)}")
print(f"\nTarget distributions:")
print(f"  fall_7d: {feature_matrix['fall_7d'].mean():.3%} positive rate")
print(f"  rth_7d:  {feature_matrix['rth_7d'].mean():.3%} positive rate")
print(f"\nNull rates (top 10 most-null features):")
null_rates = {
    c: feature_matrix[c].null_count() / feature_matrix.shape[0]
    for c in feature_cols
}
for c, rate in sorted(null_rates.items(), key=lambda x: -x[1])[:10]:
    print(f"  {c}: {rate:.1%}")

Feature matrix shape: (66312, 218)
  Meta columns:    5
  Target columns:  2
  Feature columns: 211

Target distributions:
  fall_7d: 2.420% positive rate
  rth_7d:  0.813% positive rate

Null rates (top 10 most-null features):
  weight_std_3d: 98.0%
  weight_std_7d: 95.3%
  weight_std_14d: 87.3%
  weight_mean_3d: 87.0%
  weight_min_3d: 87.0%
  weight_max_3d: 87.0%
  weight_count_3d: 87.0%
  blood_sugar_std_3d: 86.8%
  blood_sugar_std_7d: 85.5%
  blood_sugar_mean_3d: 85.1%


## 5. Temporal Split & Save

**Split strategy** (from modeling plan):
- **Train:** windows where `window_start < 2024-07-01`
- **Test:** windows where `window_start >= 2024-07-01`
- **Hold-out:** January 2025 (windows where `window_start >= 2025-01-01`)

Temporal CV folds (expanding window) will be constructed during modeling.
Here we just mark the split and save to parquet.

In [46]:
# ── Temporal split labels ─────────────────────────────────────────
SPLIT_DATE = datetime(2024, 7, 1)
HOLDOUT_DATE = datetime(2025, 1, 1)

feature_matrix = feature_matrix.with_columns(
    pl.when(pl.col("window_start") >= HOLDOUT_DATE)
    .then(pl.lit("holdout"))
    .when(pl.col("window_start") >= SPLIT_DATE)
    .then(pl.lit("test"))
    .otherwise(pl.lit("train"))
    .alias("split")
)

split_summary = (
    feature_matrix
    .group_by("split")
    .agg(
        pl.len().alias("n_windows"),
        pl.col("fall_7d").mean().alias("fall_7d_rate"),
        pl.col("rth_7d").mean().alias("rth_7d_rate"),
        pl.col("resident_id").n_unique().alias("n_residents"),
    )
    .sort("split")
)
print("Split summary:")
print(split_summary)

Split summary:
shape: (3, 5)
┌─────────┬───────────┬──────────────┬─────────────┬─────────────┐
│ split   ┆ n_windows ┆ fall_7d_rate ┆ rth_7d_rate ┆ n_residents │
│ ---     ┆ ---       ┆ ---          ┆ ---         ┆ ---         │
│ str     ┆ u32       ┆ f64          ┆ f64         ┆ u32         │
╞═════════╪═══════════╪══════════════╪═════════════╪═════════════╡
│ holdout ┆ 3612      ┆ 0.02907      ┆ 0.008029    ┆ 1007        │
│ test    ┆ 24311     ┆ 0.026202     ┆ 0.009543    ┆ 1712        │
│ train   ┆ 38389     ┆ 0.02248      ┆ 0.007242    ┆ 1749        │
└─────────┴───────────┴──────────────┴─────────────┴─────────────┘


In [57]:
# ── Save to feature store ─────────────────────────────────────────
FEATURE_STORE.mkdir(parents=True, exist_ok=True)

out_path = FEATURE_STORE / "feature_matrix_7d.parquet"
feature_matrix.write_parquet(out_path)

print(f"Saved feature matrix to {out_path}")
print(f"  Shape: {feature_matrix.shape}")
print(f"  Size:  {out_path.stat().st_size / 1e6:.1f} MB")
print(f"\nFeature columns ({len(feature_cols)}):")
for i, c in enumerate(sorted(feature_cols)):
    print(f"  {i+1:3d}. {c}")

Saved feature matrix to ../data/feature_store/feature_matrix_7d.parquet
  Shape: (66312, 218)
  Size:  6.0 MB

Feature columns (211):
    1. age_at_window
    2. blood_sugar_count_14d
    3. blood_sugar_count_3d
    4. blood_sugar_count_7d
    5. blood_sugar_max_14d
    6. blood_sugar_max_3d
    7. blood_sugar_max_7d
    8. blood_sugar_mean_14d
    9. blood_sugar_mean_3d
   10. blood_sugar_mean_7d
   11. blood_sugar_measured_14d
   12. blood_sugar_measured_3d
   13. blood_sugar_measured_7d
   14. blood_sugar_min_14d
   15. blood_sugar_min_3d
   16. blood_sugar_min_7d
   17. blood_sugar_std_14d
   18. blood_sugar_std_3d
   19. blood_sugar_std_7d
   20. bp___systolic_count_14d
   21. bp___systolic_count_3d
   22. bp___systolic_count_7d
   23. bp___systolic_max_14d
   24. bp___systolic_max_3d
   25. bp___systolic_max_7d
   26. bp___systolic_mean_14d
   27. bp___systolic_mean_3d
   28. bp___systolic_mean_7d
   29. bp___systolic_measured_14d
   30. bp___systolic_measured_3d
   31. bp___syst